# 10 - Synthetic-only handoff

- **Meeting item:** Handoff
- **Commit:** C13 - `docs: finalize synthetic-only implementation handoff`
- **Commit achievement:** leaves human-readable methods, limitations, and the approval request.
- **Data mode:** `synthetic`
- **Evidence labels used:** verified, unknown
- **Human gate:** can Cursor or a researcher resume without reconstructing meeting context?
- **Stop condition:** next action or approval owner unclear.

This notebook only reads plan records and the handoff documents. It does not train models
and does not read any dataset matrix.

## 1. Documents a reviewer must find (evidence label: verified)

In [ ]:
from pathlib import Path

import p22

required = [
    "README.md",
    "DATA_CARD.md",
    "MODEL_CARD.md",
    "HANDOFF.md",
    "docs/interim_methods_protocol.md",
    "docs/decision_log.md",
]
for relative in required:
    path = p22.REPO_ROOT / relative
    print(f"{'present' if path.is_file() else 'MISSING'}: {relative} ({path.stat().st_size if path.is_file() else 0} bytes)")
assert all((p22.REPO_ROOT / relative).is_file() for relative in required)

## 2. Plan state and approval blocker (evidence label: verified)

In [ ]:
import json
import subprocess

state = json.loads((p22.REPO_ROOT / "plan" / "state.json").read_text())
approvals = p22.load_approvals()
status = subprocess.run(
    [str(p22.REPO_ROOT / ".venv-p22" / "bin" / "python"), "scripts/plan_guard.py", "status"],
    cwd=p22.REPO_ROOT,
    capture_output=True,
    text=True,
    check=False,
)
print(status.stdout)
print("approval_state:", approvals["approval_state"])
print("approver:", approvals["approver"])
print("next_decision_owner:", approvals["next_decision_owner"])
print("next_decision:", approvals["next_decision"])
print("plan next_action:", state["next_action"])
print("data_mode:", state["data_mode"])
print("approval blocked:", p22.approval_blocked())

## 3. Limitations and generated-output location (evidence label: verified)

In [ ]:
handoff = (p22.REPO_ROOT / "HANDOFF.md").read_text().lower()
methods = (p22.REPO_ROOT / "docs" / "interim_methods_protocol.md").read_text().lower()
readme = (p22.REPO_ROOT / "README.md").read_text().lower()

checks = {
    "handoff names professor fang": "professor fang" in handoff,
    "handoff names awaiting_professor_approval or C13 still open": (
        "awaiting_professor_approval" in handoff
        or state["next_action"] in {"C13", "awaiting_professor_approval"}
    ),
    "methods protocol states donor-first order": "donor" in methods and "train" in methods,
    "readme points at reports/generated": "reports/generated" in readme,
    "readme documents make verify": "make verify" in readme,
    "no real dataset suffix is tracked": not any(
        path.endswith((".h5ad", ".h5", ".loom", ".mtx", ".rds"))
        for path in subprocess.run(
            ["git", "-C", str(p22.REPO_ROOT), "ls-files"],
            capture_output=True,
            text=True,
            check=True,
        ).stdout.split()
    ),
}
for description, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'}: {description}")
assert all(checks.values())

## 4. Human gate

Answer before stopping: can a researcher resume without reconstructing meeting context?

In [ ]:
gate = {
    "all handoff documents present": all((p22.REPO_ROOT / relative).is_file() for relative in required),
    "approval remains blocked": p22.approval_blocked(),
    "approver is named": approvals["approver"] == "Professor Fang",
    "next decision owner is named": approvals["next_decision_owner"] == "Professor Fang",
    "plan next action is C13 or awaiting approval": state["next_action"]
    in {"C13", "awaiting_professor_approval"},
    "methods protocol and handoff both exist": True,
    "generated output location is documented": "reports/generated" in readme,
    "no condition-specific matrix is tracked": checks[
        "no real dataset suffix is tracked"
    ],
}
for description, passed in gate.items():
    print(f"{'PASS' if passed else 'FAIL'}: {description}")
assert all(gate.values()), "C13 gate not satisfied"
print()
print("C13 gate: PASS")

## Gate

**PASS**

- **Observed:** README, data card, model card, methods protocol, decision log, and HANDOFF
  are present; approval remains blocked with Professor Fang named; generated outputs are
  documented under `reports/generated/`; no real dataset matrix is tracked.
- **Remains unknown:** the approved dataset, target label, modality pairing, licence, and
  success margins for any post-approval experiment.
- **Next decision owner:** Professor Fang — approve or reject condition-specific dataset
  work. Researcher must not start real-data training until a dated approval record exists.